# Real Estate Photo Enhancer

A Gradio app that makes property photos look better: it evens out lighting with CLAHE contrast correction, removes noise, and then upscales the photo 2x with Real-ESRGAN. Upload a photo and download the enhanced version.

## How to run
- Open in Google Colab. Runtime type: **T4 GPU**.
- Run the cells from top to bottom.

_Rewritten from my original Colab notebook: `real_estate_with_gradio.ipynb`._

In [ ]:
!pip install -q gradio opencv-python-headless

In [ ]:
!pip install -q realesrgan

## Fix a torchvision compatibility issue
`basicsr` (used by Real-ESRGAN) still imports `torchvision.transforms.functional_tensor`, which newer torchvision removed. This cell patches that one import line.

In [ ]:
import importlib.util
from pathlib import Path

# find_spec locates the package without importing it (importing would fail before the patch)
basicsr_dir = Path(importlib.util.find_spec("basicsr").origin).parent
deg_file = basicsr_dir / "data" / "degradations.py"
code = deg_file.read_text()
code = code.replace("torchvision.transforms.functional_tensor", "torchvision.transforms.functional")
deg_file.write_text(code)
print("Patched:", deg_file)

## Load the Real-ESRGAN model
The weights download automatically the first time.

In [ ]:
import torch
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

# RealESRGAN_x4plus: a general-purpose 4x upscaling model
model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
upsampler = RealESRGANer(
    scale=4,
    model_path="https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
    model=model,
    tile=400,          # process the image in tiles so big images fit in GPU memory
    tile_pad=10,
    pre_pad=0,
    half=torch.cuda.is_available(),  # half precision is faster on GPU
)
print("Model ready")

In [ ]:
import numpy as np
from PIL import Image

def upscale(pil_image, outscale=4):
    """Upscale a PIL image with Real-ESRGAN and return a PIL image."""
    rgb = np.array(pil_image.convert("RGB"))
    bgr = rgb[:, :, ::-1]                      # Real-ESRGAN expects BGR (OpenCV order)
    out_bgr, _ = upsampler.enhance(bgr, outscale=outscale)
    return Image.fromarray(out_bgr[:, :, ::-1])

## Brightness and noise clean-up with OpenCV
CLAHE evens out lighting in the L (lightness) channel of LAB colour space; `fastNlMeansDenoisingColored` removes grain.

In [ ]:
import cv2

def brighten_and_denoise(rgb, brightness=1.0):
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    l = cv2.convertScaleAbs(l, alpha=brightness, beta=0)
    rgb = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)
    return cv2.fastNlMeansDenoisingColored(rgb, None, 5, 5, 7, 21)

## The full pipeline + Gradio app

In [ ]:
import gradio as gr

def enhance(image, brightness, do_upscale):
    if image is None:
        return None
    cleaned = brighten_and_denoise(np.array(image.convert("RGB")), brightness)
    result = Image.fromarray(cleaned)
    if do_upscale:
        result = upscale(result, outscale=2)
    return result

gr.Interface(fn=enhance,
             inputs=[gr.Image(type="pil", label="Property photo"),
                     gr.Slider(0.8, 1.5, value=1.1, step=0.05, label="Brightness"),
                     gr.Checkbox(value=True, label="Upscale 2x with Real-ESRGAN")],
             outputs=gr.Image(type="pil", label="Enhanced photo", format="png"),
             title="Real Estate Photo Enhancer").launch(share=True)